In [ ]:
#EJEMPLO DE USO DE LA LIBRERÍA COPERNICUSMARINE PARA DESCARGAR DATOS DE LA REGIÓN DE MAGALLANES EN 2023

import copernicusmarine

ds = copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    minimum_longitude=-70.0,
    maximum_longitude=-69.0,
    minimum_latitude=-53.0,
    maximum_latitude=-52.0,
    variables=["thetao", "so", "uo", "vo"],
    start_datetime="2023-01-01",
    end_datetime="2023-12-31",
    output_directory="E:/2_Magallanes",
    output_filename="copernicus_Magallanes_Control_2023.nc"
)

In [ ]:
#ejemplo de uso de la librería xarray para convertir el cubo de datos NetCDF a un archivo CSV

import xarray as xr
import pandas as pd

# 1. Definir las rutas (usando las que definiste en tu descarga)
ruta_nc = "coleccion/copernicus_Magallanes_Control_2023.nc"
ruta_csv = "coleccion/datos_Magallanes_Control_2023.csv"

print("1. Abriendo el cubo de datos NetCDF...")
# Abrimos el archivo
ds = xr.open_dataset(ruta_nc)

print("2. Aplanando el cubo a una tabla de datos (DataFrame)...")
# Convertimos el cubo 4D a una tabla plana bidimensional
df = ds.to_dataframe()

# xarray crea índices múltiples (fecha, lat, lon). Los reseteamos para que sean columnas normales.
df = df.reset_index()

print("3. Limpiando datos nulos (tierra firme)...")
# Los píxeles que caen sobre tierra no tienen temperatura del agua (son NaN).
# Los eliminamos para que el CSV no pese gigabytes innecesariamente.
df_limpio = df.dropna(subset=['thetao', 'so', 'uo', 'vo'], how='all')

print("4. Guardando en el disco duro...")
# Exportamos a CSV
df_limpio.to_csv(ruta_csv, index=False)

print("\n¡Conversión Exitosa!")
print(f"Registros totales en el CSV: {len(df_limpio)}")
print("\nVista previa de las primeras 5 filas:")
print(df_limpio[['time', 'latitude', 'longitude', 'depth', 'thetao', 'so']].head())

# Cerramos el archivo original para liberar memoria RAM
ds.close()

In [ ]:
#ejemplo de uso de la librería pandas y matplotlib para graficar los datos del CSV corregido

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Cargar tu CSV local
ruta_csv = "coleccion/datos_Magallanes_Control_2023.csv"
df = pd.read_csv(ruta_csv)

# 2. EL FILTRO DE INGENIERÍA (Arreglando el desastre dimensional)
# A) Aislar solo la capa superficial (la profundidad mínima del dataset)
profundidad_superficie = df['depth'].min()
df_superficie = df[df['depth'] == profundidad_superficie]

# B) Colapsar el mapa espacial: Promediar todos los píxeles de esa gran caja por cada día
df_limpio = df_superficie.groupby('time').mean(numeric_only=True).reset_index()

# 3. Asegurar que Python entienda la fecha
df_limpio['time'] = pd.to_datetime(df_limpio['time'])

# --- De aquí en adelante, usa df_limpio en vez de df para tus gráficos ---

fig, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
fig.suptitle('Análisis Oceanográfico 2023 (Corregido) - Superficie', fontsize=16, fontweight='bold', y=0.92)

# Gráfico 1: Temperatura
axs[0].plot(df_limpio['time'], df_limpio['thetao'], color='firebrick', linewidth=1.5)
axs[0].set_ylabel('Temperatura (°C)', fontweight='bold')
axs[0].set_title('Temperatura Superficial del Mar (thetao)', loc='left')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].axhline(y=10.5, color='red', linestyle=':', label='Umbral referencial brote')
axs[0].legend(loc='upper right')

# Gráfico 2: Salinidad
axs[1].plot(df_limpio['time'], df_limpio['so'], color='teal', linewidth=1.5)
axs[1].set_ylabel('Salinidad (PSU)', fontweight='bold')
axs[1].set_title('Salinidad Superficial (so)', loc='left')
axs[1].grid(True, linestyle='--', alpha=0.6)

# Gráfico 3: Corriente U
axs[2].plot(df_limpio['time'], df_limpio['uo'], color='steelblue', linewidth=1)
axs[2].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[2].set_title('Corriente Este-Oeste (uo)', loc='left')
axs[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[2].grid(True, linestyle='--', alpha=0.6)

# Gráfico 4: Corriente V
axs[3].plot(df_limpio['time'], df_limpio['vo'], color='purple', linewidth=1)
axs[3].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[3].set_title('Corriente Norte-Sur (vo)', loc='left')
axs[3].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[3].grid(True, linestyle='--', alpha=0.6)

axs[3].xaxis.set_major_locator(mdates.MonthLocator())
axs[3].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.xlabel('Meses del Año 2023', fontweight='bold', fontsize=12)
plt.setp(axs[3].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
#para el viento

import copernicusmarine
import time
import os

# 1. Configuración Maestra de los Centros
centros = {
    "110557": {
        "lat": -45.3214,
        "lon": -73.3567,
        "directorio": "coleccion/Historico_110557"
    },
    "110069": {
        "lat": -53.2107,
        "lon": -72.8302,
        "directorio": "coleccion/Historico_110069"
    }
}

# Parámetros espaciales y de la API
buffer = 0.3 
dataset_bgc = "cmems_obs-wind_glo_phy_my_l3-metopa-ascat-asc-0.125deg_P1D-i"

print("Iniciando extracción multizona de viento...")

# 2. Ciclo principal de extracción
for id_centro, coords in centros.items():
    print(f"\n=========================================")
    print(f"Procesando Centro: {id_centro}")
    print(f"=========================================")
    
    # Asegurar que la carpeta exista
    os.makedirs(coords["directorio"], exist_ok=True)
    
    # Calcular bounding box
    min_lon = coords["lon"] - buffer
    max_lon = coords["lon"] + buffer
    min_lat = coords["lat"] - buffer
    max_lat = coords["lat"] + buffer

    for anio in range(2020, 2026):
        fecha_inicio = f"{anio}-01-01"
        fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
        
        nombre_archivo = f"{id_centro}_BGC_{anio}_Micro_Viento.nc"
        ruta_completa = os.path.join(coords["directorio"], nombre_archivo)
        
        # Validación para saltar años ya descargados (Esto reemplaza a overwrite_output_data)
        if os.path.exists(ruta_completa):
            print(f"[SKIP] El archivo {nombre_archivo} ya existe en disco.")
            continue

        print(f"-> Extrayendo año {anio}...")
        
        try:
            copernicusmarine.subset(
                dataset_id=dataset_bgc,
                variables=["eastward_wind", "northward_wind"],
                start_datetime=fecha_inicio,
                end_datetime=fecha_fin,
                minimum_longitude=min_lon,
                maximum_longitude=max_lon,
                minimum_latitude=min_lat,
                maximum_latitude=max_lat,
                minimum_depth=0.4941, 
                maximum_depth=0.51,
                output_directory=coords["directorio"],
                output_filename=nombre_archivo
            ) 
            print(f"[OK] {nombre_archivo} guardado correctamente.")
            time.sleep(3)
            
        except Exception as e:
            print(f"[ERROR] Fallo crítico en el año {anio} para el centro {id_centro}: {e}")

print("\nPipeline de extracción BGC finalizado.")

In [ ]:
#estructuracion interesante para luego
import xarray as xr
import pandas as pd
import os

# 1. Configuración de los Centros
centros = {
    "110557": {
        "directorio": "coleccion/Historico_110557"
    },
    "110069": {
        "directorio": "coleccion/Historico_110069"
    }
}

print("Iniciando aplanamiento dimensional y validación de variables BGC...\n")

for id_centro, info in centros.items():
    lista_df_anuales = []
    
    print(f"=========================================")
    print(f"Procesando Centro: {id_centro}")
    print(f"=========================================")
    
    for anio in range(2020, 2026):
        nombre_archivo = f"{id_centro}_BGC_{anio}_Micro_Bioquimica.nc"
        ruta_completa = os.path.join(info["directorio"], nombre_archivo)
        
        if os.path.exists(ruta_completa):
            # Abrir el tensor tridimensional
            ds_bgc = xr.open_dataset(ruta_completa)
            
            # Identificar qué dimensiones existen para promediarlas (colapso espacial)
            dims_to_drop = [d for d in ['latitude', 'longitude', 'depth'] if d in ds_bgc.dims]
            
            # Promediar y convertir a formato tabular (DataFrame)
            df_anio = ds_bgc.mean(dim=dims_to_drop).to_dataframe().reset_index()
            
            # Filtrar solo las columnas que nos interesan
            columnas_necesarias = ['time', 'chl', 'o2']
            df_anio = df_anio[[c for c in columnas_necesarias if c in df_anio.columns]]
            
            lista_df_anuales.append(df_anio)
            ds_bgc.close()
        else:
            print(f"[AVISO] Archivo del año {anio} no encontrado para el centro {id_centro}.")

    # Si logramos extraer datos, los consolidamos
    if lista_df_anuales:
        df_final = pd.concat(lista_df_anuales, ignore_index=True)
        
        # Renombrar columnas para el estándar del proyecto
        df_final.rename(columns={
            'time': 'Fecha',
            'chl': 'Clorofila_mg_m3',
            'o2': 'Oxigeno_mmol_m3'
        }, inplace=True)
        
        # Limpieza de fechas nulas y ordenamiento cronológico
        df_final.dropna(subset=['Fecha'], inplace=True)
        df_final.sort_values('Fecha', inplace=True)
        
        # Guardar como un dataset separado
        ruta_csv = os.path.join(info["directorio"], f"Bioquimica_Independiente_{id_centro}.csv")
        df_final.to_csv(ruta_csv, index=False, encoding='utf-8')
        
        print(f"[ÉXITO] Matriz química guardada en: {ruta_csv}")
        
        # --- VALIDACIÓN ESTADÍSTICA (Para ver los resultados) ---
        print("\nResumen Estadístico de los Datos (QA/QC):")
        # El método describe() nos dará la media, min y max para descartar anomalías
        print(df_final[['Clorofila_mg_m3', 'Oxigeno_mmol_m3']].describe().round(3))
        print("\nMuestra de los primeros 3 días:")
        print(df_final.head(3).to_string(index=False))
        print("\n")
        
    else:
        print(f"[ERROR] No se pudo generar el dataset para el centro {id_centro}.\n")